# Partitioning Analysis — 3-axis persona (CircuitNet-N28)

Companion to `partitioning_analysis.ipynb`. Same two "main" strategies
(hierarchical deterministic and data-driven k-prototypes), but the persona
axis is expanded from `(design × clock)` to `(design × clock × utilization)`.

| id | strategy | client count |
| -- | -------- | ------------ |
| P1c | Hierarchical deterministic: `design × clock_bin × util_bin` | 6 × 2 × 2 = **24** |
| P3c | k-prototypes on weighted mixed-type features, `k = 24` | 24 |

Both `clock_ns` and `utilization` are already in the P3 weight schema
(w = 1.5 each, via `log_frequency` and `utilization`), so bumping k from 12
to 24 asks k-prototypes to split each of the 12 (design × clock-tier) cells
roughly along the utilization axis — the same cross-product P1c constructs
deterministically. Comparing them via ARI/NMI + confusion matrix is the
cleanest test of whether the natural metadata clusters match the hand-drawn
3-axis personas.

# Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 11,
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5',
    'figure.facecolor': 'white',
})

sys.path.insert(0, os.path.abspath('.'))
from partitioning import (
    HierarchicalPersonaPartitioner,
    KPrototypesPartitioner,
)

FEATURE_DIR = '../routability_ir_drop_prediction/training_set_N28/DRC/feature'
OUT_DIR = './partition_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('Imports OK.')

# Load metadata + derive features

Same parser and derivations as the main partitioning notebook so both P1c
(hierarchical) and P3c (k-prototypes) run on identical inputs.

In [ ]:
def parse_sample_name(filename: str) -> dict:
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    if len(parts) < 7:
        raise ValueError(f'Cannot parse: {filename}')
    for expected, token in zip(['c','u','m','p','f'], parts[-5:]):
        if not token.startswith(expected):
            raise ValueError(f'Bad token {token} in {filename}')
    c, u, m, p, f = parts[-5:]
    return {
        'design_name':      '-'.join(parts[:-6]),
        'macro_count':      parts[-6],
        'clock_ns':         float(c[1:]),
        'utilization':      float(u[1:]),
        'macro_placement':  m[1:],
        'power_mesh':       p[1:],
        'filler_insertion': f[1:],
        'filename':         filename,
    }

files = sorted(f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy'))
df_meta = pd.DataFrame([parse_sample_name(f) for f in files])
df_meta.loc[~df_meta['macro_placement'].isin(['1','2','3','4']), 'macro_placement'] = '1'

SIZE_MAP = {
    'zero-riscy-a': 'small', 'zero-riscy-b': 'small',
    'RISCY-a':      'small', 'RISCY-b':      'small',
    'RISCY-FPU-a':  'small', 'RISCY-FPU-b':  'small',
    'OpenC910-1':   'medium', 'Vortex-small': 'medium',
    'Vortex-large': 'large',  'NVDLA-small':  'large', 'NVDLA-large': 'large',
}
FILLER_MAP = {'0': 'after_routing', '1': 'after_placement'}

df_meta['size_class']    = df_meta['design_name'].map(SIZE_MAP).fillna('small')
df_meta['log_frequency'] = np.log10(1000.0 / df_meta['clock_ns'].astype(float))
df_meta['filler']        = df_meta['filler_insertion'].map(FILLER_MAP).fillna(df_meta['filler_insertion'])
df_meta['aspect_ratio']  = 1.0

print(f'Samples: {len(df_meta)}   Designs: {df_meta["design_name"].nunique()}')
print(f'clock_ns    unique: {sorted(df_meta["clock_ns"].unique().tolist())}')
print(f'utilization unique: {sorted(df_meta["utilization"].unique().tolist())}')

# Config

`N_CLIENTS = 6 × 2 × 2 = 24`:

- 6 unique `design_name` values (RISCY-a/b, RISCY-FPU-a/b, zero-riscy-a/b),
- 2 quantile bins of `clock_ns` (splits the {2, 5, 20} ns values into
  low-period / high-period ≈ high-freq / low-freq),
- 2 quantile bins of `utilization` (splits the {0.70, 0.75, 0.80, 0.85, 0.90}
  values into low-density / high-density).

24 clients on ~10k samples gives ~425 samples/client on average — still
usable for the DRC CNN, but note that some cells (e.g. rare
clock/util pairs) will be smaller.

In [ ]:
N_DESIGNS       = df_meta['design_name'].nunique()
N_CLOCK_BINS    = 2
N_UTIL_BINS     = 2
N_CLIENTS       = N_DESIGNS * N_CLOCK_BINS * N_UTIL_BINS
SEED            = 42
print(f'N_CLIENTS = {N_DESIGNS} designs x {N_CLOCK_BINS} clock bins x {N_UTIL_BINS} util bins = {N_CLIENTS}')

# Scoring helpers

Identical to `partitioning_analysis.ipynb`. Repeated here so this notebook is
self-contained.

In [ ]:
FACTOR_COLS = [
    'design_name', 'clock_ns', 'utilization',
    'macro_placement', 'power_mesh', 'filler_insertion',
]

global_props = {c: df_meta[c].astype(str).value_counts(normalize=True) for c in FACTOR_COLS}


def per_client_composition(parts, col):
    cols = []
    for i, p in enumerate(parts):
        vc = p[col].astype(str).value_counts(normalize=True)
        vc.name = f'c{i}'
        cols.append(vc)
    return pd.concat(cols, axis=1).fillna(0.0).sort_index()


def js_divergence_to_global(parts, col, global_dist):
    all_levels = sorted(set(global_dist.index) | set().union(
        *[set(p[col].astype(str).unique()) for p in parts]
    ))
    g = global_dist.reindex(all_levels).fillna(0.0).to_numpy()
    g = g / max(g.sum(), 1e-12)
    out = []
    for p in parts:
        vc = p[col].astype(str).value_counts(normalize=True)
        v = vc.reindex(all_levels).fillna(0.0).to_numpy()
        if v.sum() == 0:
            out.append(np.nan)
        else:
            v = v / v.sum()
            out.append(float(jensenshannon(v, g, base=2) ** 2))
    return out


def summarize(parts, name, factor_cols=FACTOR_COLS):
    sizes = [len(p) for p in parts]
    print(f'\n=== {name} ===')
    print(f'  clients            : {len(parts)}')
    print(f'  size min/median/max: {min(sizes)} / {int(np.median(sizes))} / {max(sizes)}')
    imb = max(sizes) / max(min(sizes), 1)
    print(f'  size imbalance     : max/min = {imb:.2f}x')
    js = {}
    for col in factor_cols:
        js[col] = float(np.nanmean(js_divergence_to_global(parts, col, global_props[col])))
    for col, v in js.items():
        print(f'  mean JS({col:20s}) = {v:.3f}')
    return {'name': name, 'sizes': sizes, 'imbalance': imb, 'js_mean': js}


def plot_partition_composition(parts, name, cols=('design_name','clock_ns','utilization')):
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 6))
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        comp = per_client_composition(parts, col)
        levels = comp.index.tolist()
        n_clients = comp.shape[1]
        cmap = mcm.get_cmap('tab20', max(len(levels), 3))
        bottom = np.zeros(n_clients)
        for j, lvl in enumerate(levels):
            h = comp.iloc[j].to_numpy()
            ax.bar(range(n_clients), h, bottom=bottom,
                   color=cmap(j % cmap.N), label=str(lvl),
                   alpha=0.9, edgecolor='white', linewidth=0.5)
            bottom += h
        ax.set_xticks(range(n_clients))
        ax.set_xticklabels([f'c{i}' for i in range(n_clients)], rotation=60, ha='right', fontsize=7)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('within-client proportion')
        ax.set_title(f'{name}: P({col} | client)')
        if len(levels) <= 14:
            ax.legend(fontsize=7, loc='center left', bbox_to_anchor=(1.0, 0.5))
    plt.tight_layout()
    plt.show()


def sample_to_client(parts, key='filename'):
    m = {}
    for i, p in enumerate(parts):
        for k in p[key]:
            m[k] = i
    return m


def align_labels(reference_parts, other_parts, key='filename'):
    r = sample_to_client(reference_parts, key)
    o = sample_to_client(other_parts, key)
    keys = [k for k in r if k in o]
    return (np.array([r[k] for k in keys]),
            np.array([o[k] for k in keys]))


results = {}

# P1c — Hierarchical deterministic (design × clock_bin × util_bin)

Uses the multi-axis form of `HierarchicalPersonaPartitioner`:

```python
HierarchicalPersonaPartitioner(
    n_partitions=24,
    design_col='design_name',
    persona_col=['clock_ns', 'utilization'],
    n_persona_bins=[2, 2],
)
```

Cell key format: `<design>|clock_ns<i>|utilization<j>`.

Because both persona axes use quantile bins and the underlying design-space
sweep is fully factorial (every design × every clock × every utilization),
the 24 cells should end up roughly evenly populated.

In [ ]:
p1c = HierarchicalPersonaPartitioner(
    n_partitions=N_CLIENTS,
    design_col='design_name',
    persona_col=['clock_ns', 'utilization'],
    n_persona_bins=[N_CLOCK_BINS, N_UTIL_BINS],
    bin_method='quantile',
    seed=SEED,
).partition(df_meta)

print('P1c cells (design|clock_bin|util_bin, sample count, clock/util unique):')
for i, part in enumerate(p1c):
    print(f'  c{i:2d}  {part["_client_cell"].iloc[0]:45s} n={len(part):5d}  '
          f'clock={sorted(part["clock_ns"].unique())}  '
          f'util={sorted(part["utilization"].unique())}')

results['P1c_hier3'] = summarize(p1c, 'P1c Hierarchical (design x clock x util)')
plot_partition_composition(p1c, 'P1c Hierarchical',
                           cols=('design_name','clock_ns','utilization'))

# P3c — k-prototypes at k = 24

Same weight schema as `partitioning_analysis.ipynb`:
`design_name` (2.0), `size_class` (1.0), `log_frequency` (1.5),
`utilization` (1.5), `aspect_ratio` (1.0), `power_mesh` (0.75),
`macro_placement` (0.75), `filler` (0.5). `size_class` and `aspect_ratio`
are constant on N28 and dropped automatically.

At k=24, k-means has enough room to spend one cluster on every plausible
(design × clock-tier × util-tier) cell. Whether it does so — vs. e.g.
carving out something driven by the low-weight flow attributes — is what
the alignment check in the next section measures.

In [ ]:
FEATURE_SPECS = [
    ('design_name',      'nominal',  2.00),
    ('size_class',       'ordinal',  1.00),
    ('log_frequency',    'interval', 1.50),
    ('utilization',      'interval', 1.50),
    ('aspect_ratio',     'interval', 1.00),
    ('power_mesh',       'nominal',  0.75),
    ('macro_placement',  'nominal',  0.75),
    ('filler',           'nominal',  0.50),
]
ORDINAL_ORDERS = {'size_class': ['small', 'medium', 'large']}

p3c_part = KPrototypesPartitioner(
    n_partitions=N_CLIENTS,
    feature_specs=FEATURE_SPECS,
    ordinal_orders=ORDINAL_ORDERS,
    seed=SEED,
    n_init=20,
)
p3c = p3c_part.partition(df_meta)

print(f'Dropped features (missing/constant): {p3c_part.dropped_features_}')
print(f'Encoded matrix shape: {p3c_part.encoded_matrix_.shape}')
print(f'k-means inertia: {p3c_part.kmeans_.inertia_:.2f}')

results['P3c_kproto24'] = summarize(p3c, f'P3c K-prototypes (k={N_CLIENTS})')
plot_partition_composition(p3c, 'P3c K-prototypes',
                           cols=('design_name','clock_ns','utilization'))

# P1c ↔ P3c alignment (3-axis persona validation)

Same test as in the main partitioning notebook, but on the 3-axis grid.
If ARI/NMI are noticeably higher here than in the 2-axis comparison, adding
the utilization split into the hand-drawn personas is a good match for what
the data-driven clustering was already doing implicitly. If they're the same
or worse, utilization isn't providing enough independent separation to
justify the extra client count.

In [ ]:
labels_p1c, labels_p3c = align_labels(p1c, p3c, key='filename')
ari = adjusted_rand_score(labels_p1c, labels_p3c)
nmi = normalized_mutual_info_score(labels_p1c, labels_p3c)
print(f'ARI(P1c, P3c) = {ari:.3f}')
print(f'NMI(P1c, P3c) = {nmi:.3f}')

cell_names = [p1c[i]['_client_cell'].iloc[0] for i in range(len(p1c))]
cm = pd.crosstab(
    pd.Series(labels_p1c, name='P1c (design|clock|util)').map(dict(enumerate(cell_names))),
    pd.Series(labels_p3c, name='P3c (cluster)'),
)
print()
print(cm)

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(cm.values, cmap='Blues', aspect='auto')
ax.set_xlabel('P3c cluster id')
ax.set_ylabel('P1c client (design | clock_bin | util_bin)')
ax.set_xticks(range(cm.shape[1]))
ax.set_xticklabels(cm.columns)
ax.set_yticks(range(cm.shape[0]))
ax.set_yticklabels(cm.index, fontsize=7)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        v = cm.values[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center', fontsize=6,
                    color='white' if v > cm.values.max() / 2 else 'black')
ax.set_title(f'P1c vs P3c confusion  (ARI={ari:.3f}, NMI={nmi:.3f})')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Summary

Side-by-side size imbalance and per-factor JS divergence for the two 3-axis
strategies. Interpretation:

- **JS on `design_name` / `clock_ns` / `utilization`** should all be high
  for both strategies — the whole point of the 3-axis grid is to concentrate
  each of them into specific clients.
- **JS on flow columns** (`macro_placement`, `power_mesh`,
  `filler_insertion`) should stay low: neither the hand-drawn cells nor the
  k-prototypes weighting is asking the flow attributes to separate anything.

In [ ]:
rows = []
for name, res in results.items():
    row = {
        'strategy':   name,
        'n_clients':  len(res['sizes']),
        'imbalance':  round(res['imbalance'], 2),
        'min_size':   min(res['sizes']),
        'max_size':   max(res['sizes']),
    }
    for col in FACTOR_COLS:
        row[f'JS[{col}]'] = round(res['js_mean'][col], 3)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('strategy')
print(summary_df.to_string())

primary_cols = [c for c in summary_df.columns if c.startswith('JS[') and
                any(k in c for k in ('design_name','clock_ns','utilization'))]
flow_cols    = [c for c in summary_df.columns if c.startswith('JS[') and
                any(k in c for k in ('macro_placement','power_mesh','filler_insertion'))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, group, title in [
    (axes[0], primary_cols, 'JS on PRIMARY / PERSONA axes  (should be HIGH)'),
    (axes[1], flow_cols,    'JS on FLOW axes  (should stay LOW)'),
]:
    sub = summary_df[group]
    x = np.arange(len(sub.index))
    width = 0.8 / max(len(group), 1)
    for j, col in enumerate(group):
        ax.bar(x + j*width, sub[col].values, width=width,
               label=col.replace('JS[','').rstrip(']'))
    ax.set_xticks(x + width * (len(group) - 1) / 2)
    ax.set_xticklabels(sub.index, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('Mean JS divergence')
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Save partitions

Written to `./partition_outputs/`; one CSV per (strategy, client) listing
the sample filenames that belong to that client.

In [ ]:
def save_partitions(parts, prefix):
    for i, p in enumerate(parts):
        p[['filename']].to_csv(
            os.path.join(OUT_DIR, f'{prefix}_client_{i:02d}.csv'), index=False
        )
    print(f'  saved {len(parts)} files: {prefix}_client_*.csv')

save_partitions(p1c,  'P1c_hier3axis')
save_partitions(p3c,  'P3c_kproto24')
print(f'All partitions written to {OUT_DIR}/')